Author: **Dongyuan Gao**

Course: HSLU Computer Vision — Lecture 3 Project

Based on the style of the lecturer's notebooks by *Safouane El Ghazouali* (TOELT LLC / HSLU).

# -----  -----  -----  -----  -----  -----  -----  -----

# YOLO26 + CLIP Car Brand Recognition on Video

This notebook loads a fine-tuned **YOLO26** detector and a **CLIP linear probe** (20 car brands), then runs inference on dashcam videos frame-by-frame.

For every detected **car**, the corresponding bounding box is cropped and passed to the CLIP model to predict the most likely brand. Truck detections are intentionally **not** sent to the brand classifier because the linear probe was trained exclusively on car images — truck crops are out-of-distribution and would yield miscalibrated predictions.

The annotated output video is saved for further processing in Stage 3 (VLM captions).

### What You'll Learn
- Loading fine-tuned YOLO and CLIP models.
- Processing video frame-by-frame with YOLO detection.
- Cropping detected vehicles and running CLIP brand classification.
- Annotating frames with brand labels and confidence scores.
- Saving annotated output video for Stage 3 VLM overlay.

# Running on DGX via VS Code Remote

Project directory on DGX: `/home/dongyuan/Desktop/computer_vision`

Typical flow:
- Connect to the DGX with VS Code Remote - SSH.
- Open this notebook **on the remote machine** (so paths refer to DGX storage).
- Use a conda env or venv with PyTorch + CUDA already installed.
- Keep datasets on DGX local storage (faster than network mounts).

# Environment Setup (DGX)

Install Ultralytics (YOLO), Roboflow (dataset download), and OpenCV.

On a DGX, you typically already have a CUDA-enabled PyTorch in your conda env.
If you do not, create or activate your environment before running the install below.

In [78]:
!pip install -q ultralytics roboflow opencv-python
!pip install open-clip-torch
!pip install torch


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### Optional: Ollama Python Client (local VLM captions)

If you want to run the VLM overlay cell later, install the **Python client** in your environment.
The Ollama server itself is installed and run in the terminal (system-level).

Example install (terminal or notebook cell): `pip install ollama`

### Import Libraries & Check GPU

On the DGX you should see `cuda` and at least one visible GPU.
If it prints `cpu`, your environment is missing CUDA-enabled PyTorch or no GPU is visible.

In [79]:
from ultralytics import YOLO
from roboflow import Roboflow
import torch
import os, glob, yaml
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import torch.nn as nn
import open_clip
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

# Quick GPU visibility check on DGX
!nvidia-smi -L

# Explanation
# - device: tells YOLO where to run (GPU is ~30x faster than CPU).
# - Ultralytics auto-uses this device unless we override it.

Using device: cuda
PyTorch version: 2.11.0+cu130
GPU 0: NVIDIA GB10 (UUID: GPU-0b6645ac-fb60-3d81-c925-eb574014af92)


# Dataset on the DGX (Roboflow or Local Path)

You can either download with Roboflow **on the DGX** or point to a dataset that is already on DGX storage.

**Option A (Roboflow download on DGX):**
1. Go to https://public.roboflow.com/object-detection/self-driving-car
2. Click **Download Dataset** → pick **YOLOv8** format (compatible with v10).
3. Roboflow shows you a **personalized snippet** with your API key — paste it in the next cell.

**Option B (Dataset already on DGX):**
- Set the `DATASET_DIR` path below to the folder that contains `data.yaml`, `train/`, `valid/`, `test/`.

**Note (local path):** If you set `USE_ROBOFLOW = False`, this notebook looks for the dataset in `./Self-Driving-Car-3` or `./self-driving-car`. You can also override with an environment variable, e.g. `export DATASET_DIR=/path/to/dataset`.


In [80]:
# Set this to False if the dataset is already on DGX storage
USE_ROBOFLOW = False

# If USE_ROBOFLOW is False, set the local dataset folder on DGX
def resolve_dataset_dir() -> str:
    env_path = os.getenv("DATASET_DIR")
    if env_path:
        return env_path
    candidates = [
        os.path.join(os.getcwd(), "Self-Driving-Car-3"),
        os.path.join(os.getcwd(), "self-driving-car"),
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    raise FileNotFoundError(
        "Dataset folder not found. Set DATASET_DIR or place dataset at ./Self-Driving-Car-3 or ./self-driving-car"
    )

if USE_ROBOFLOW:
    # ---- PASTE YOUR ROBOFLOW SNIPPET HERE ----
    rf = Roboflow(api_key="YOUR_API_KEY")
    project = rf.workspace("roboflow-gw7yv").project("self-driving-car")
    dataset = project.version(3).download("yolov8")
    dataset_location = dataset.location
else:
    DATASET_DIR = resolve_dataset_dir()
    dataset_location = DATASET_DIR

data_yaml = os.path.join(dataset_location, "data.yaml")
print(f"Dataset location: {dataset_location}")
print(f"data.yaml: {data_yaml}")

# Explanation
# - dataset_location: absolute path to the dataset folder on DGX
# - data.yaml lists class names and the train/valid/test paths YOLO needs

Dataset location: /home/dongyuan/Desktop/computer_vision/Self-Driving-Car-3
data.yaml: /home/dongyuan/Desktop/computer_vision/Self-Driving-Car-3/data.yaml


## Load CLIP model and linear probe

This runtime notebook supports two modes:

- current local repo layout (`weights/clip/linear_probe`, `weights/yolo`, `original_videos`, `runs_output`),
- older or alternate layouts via environment variables or fallback path detection.

Optional environment overrides:

- `PROBE_DIR` for the CLIP linear probe directory,
- `YOLO_WEIGHTS` for the YOLO weight file,
- `INPUT_VIDEO` for the input video path,
- `OUTPUT_DIR` for the output video directory.

In [81]:
# ============================================================
# Load CLIP model for car brand classification
# ============================================================

import open_clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED,
    device=DEVICE
)

clip_model.eval()

# Load your trained linear probe
# Example: sklearn LogisticRegression / LinearSVC / etc.
# linear_probe = joblib.load("car_brand_linear_probe.pkl")
# ============================================================
# Load CLIP model + PyTorch linear probe
# ============================================================

import json
from pathlib import Path
import torch.nn as nn
import open_clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROBE_DIR = Path("weights/clip/linear_probe")

# ------------------------------------------------------------
# Load config
# ------------------------------------------------------------

with open(PROBE_DIR / "config.json", "r") as f:
    config = json.load(f)

MODEL_NAME = config["clip_model"]
PRETRAINED = config["pretrained"]
embed_dim = config["embed_dim"]
n_classes = config["n_classes"]

# ------------------------------------------------------------
# Load class names
# ------------------------------------------------------------

with open(PROBE_DIR / "class_names.json", "r") as f:
    class_names = json.load(f)

print("Classes:", class_names)

# ------------------------------------------------------------
# Load CLIP model
# ------------------------------------------------------------

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED,
    device=DEVICE
)

clip_model.eval()

# ------------------------------------------------------------
# Rebuild linear probe architecture
# ------------------------------------------------------------

linear_probe = nn.Linear(embed_dim, n_classes)

# ------------------------------------------------------------
# Load trained weights
# ------------------------------------------------------------

state_dict = torch.load(
    PROBE_DIR / "linear_probe_weights.pt",
    map_location=DEVICE
)

linear_probe.load_state_dict(state_dict)

linear_probe.to(DEVICE)
linear_probe.eval()

print("CLIP + linear probe loaded")

Classes: ['Audi', 'BMW', 'Chevrolet', 'Citroen', 'Dacia', 'Fiat', 'Ford', 'Honda', 'Hyundai', 'Kia', 'Mercedes', 'Nissan', 'Opel', 'Peugeot', 'Renault', 'Seat', 'Skoda', 'Tofaş', 'Toyota', 'Volkswagen']
CLIP + linear probe loaded


In [82]:
# ============================================================
# Predict car brand from cropped image
# ============================================================

import torch.nn.functional as F

def predict_car_brand(crop_bgr):

    # OpenCV BGR -> RGB
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    # Convert to PIL
    pil_image = Image.fromarray(crop_rgb)

    # CLIP preprocessing
    image_tensor = clip_preprocess(pil_image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        # ----------------------------------------------------
        # Image embedding
        # ----------------------------------------------------

        features = clip_model.encode_image(image_tensor)

        # SAME normalization as training
        features = F.normalize(features, dim=-1)

        # ----------------------------------------------------
        # Linear probe prediction
        # ----------------------------------------------------

        logits = linear_probe(features)

        probs = torch.softmax(logits, dim=1)

        confidence, pred_idx = probs.max(dim=1)

        confidence = confidence.item()
        pred_idx = pred_idx.item()

    brand_name = class_names[pred_idx]

    return brand_name, confidence

## Load yolo fine-tuned model

In [83]:
model = YOLO('weights/yolo/best.pt')

# Part 2 — Video Demo (DGX Path Input)

Place a dashcam clip on the DGX (scp it from your Mac if needed).
The code below processes every frame and **saves an annotated output video** on the DGX.

In [84]:
# ============================================================
# YOLO + CLIP Car Brand Recognition on Video
# ============================================================

import cv2
import os
from pathlib import Path
from tqdm import tqdm

# ------------------------------------------------------------
# Input video
# ------------------------------------------------------------

video_path = "original_videos/dashcam.mp4"

assert os.path.exists(video_path), "Video path not found"

# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------

output_dir = Path("runs_output/detect/clip_predict")
output_dir.mkdir(parents=True, exist_ok=True)

output_video_path = output_dir / "annotated_video.mp4"

# ------------------------------------------------------------
# Open video
# ------------------------------------------------------------

cap = cv2.VideoCapture(video_path)

assert cap.isOpened(), "Could not open video"

# Video properties
# Keep fps as float so 29.97 / 23.976 sources are not silently rounded down to 29 / 23,
# which would otherwise misalign Step 3's frame-index seeking and caption gating.
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"FPS: {fps}")
print(f"Resolution: {width}x{height}")
print(f"Frames: {frame_count}")

# ------------------------------------------------------------
# Video writer
# ------------------------------------------------------------

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(output_video_path),
    fourcc,
    float(fps),
    (width, height)
)

# Guard: if the codec is unavailable (rare on DGX, common in stripped opencv-python-headless
# builds), VideoWriter returns silently and write() becomes a no-op, leaving a 0-byte mp4
# that Step 3's auto-discovery would later treat as a valid annotated video.
if not writer.isOpened():
    cap.release()
    writer.release()
    if output_video_path.is_file():
        try:
            output_video_path.unlink()
        except OSError:
            pass
    raise RuntimeError(
        f"cv2.VideoWriter failed to open with fourcc 'mp4v' for {output_video_path}. "
        "The OpenCV build is missing the required codec."
    )

# ------------------------------------------------------------
# Drawing style (amber chip + black text — high contrast on most scenes)
# ------------------------------------------------------------

LABEL_FONT       = cv2.FONT_HERSHEY_DUPLEX
LABEL_THICK      = 1
# Font auto-shrinks to fit each box width, clamped between MIN and MAX.
# Dashcam cars are often ~45-60 px wide, so a fixed 0.7 scale overflowed
# and made neighbouring labels collide — hence the adaptive range.
LABEL_SCALE_MAX  = 0.6
LABEL_SCALE_MIN  = 0.35
BOX_COLOR        = (0, 200, 255)   # BGR amber/orange (default)
GREEN_COLOR      = (0, 200, 0)     # BGR green -> trafficLight-Green / GreenLeft
RED_COLOR        = (0, 0, 255)     # BGR red   -> trafficLight-Red
TEXT_COLOR       = (0, 0, 0)

# Only draw the brand label when CLIP is reasonably confident.
# With 20 brand classes, a softmax near 5 % is random-chance.
# 0.15 captures distant / rear-view crops that scored below the old 0.3 gate.
BRAND_CONF_THRESHOLD = 0.3

# Minimum detection size (px) before we bother running CLIP.
# clip_preprocess() already resizes any crop up to 224x224, so this is
# just a safety floor to skip degenerate / near-zero-pixel boxes.
MIN_CROP_SIZE = 30

def draw_label(img, x1, y1, x2, y2, text, box_color=BOX_COLOR):
    cv2.rectangle(img, (x1, y1), (x2, y2), box_color, 2)

    box_w = max(1, x2 - x1)

    # Pick the largest scale (<= MAX) whose text still fits the box width;
    # never go below MIN so it stays legible on tiny boxes.
    scale = LABEL_SCALE_MAX
    (tw, th), bl = cv2.getTextSize(text, LABEL_FONT, scale, LABEL_THICK)
    if tw > box_w - 6:
        scale = max(LABEL_SCALE_MIN, scale * (box_w - 6) / tw)
        (tw, th), bl = cv2.getTextSize(text, LABEL_FONT, scale, LABEL_THICK)

    chip_h = th + bl + 6
    # Prefer above the box; if too close to the top, draw inside the box.
    if y1 - chip_h >= 0:
        chip_y1, chip_y2 = y1 - chip_h, y1
        text_y = chip_y2 - bl - 2
    else:
        chip_y1, chip_y2 = y1, min(img.shape[0], y1 + chip_h)
        text_y = chip_y1 + th + 2

    chip_x1 = x1
    # Chip is sized to the (now-fitted) text and clamped to the frame edge.
    chip_x2 = min(img.shape[1], x1 + tw + 8)
    cv2.rectangle(img, (chip_x1, chip_y1), (chip_x2, chip_y2), box_color, -1)
    cv2.putText(img, text, (chip_x1 + 4, text_y),
                LABEL_FONT, scale, TEXT_COLOR, LABEL_THICK, cv2.LINE_AA)

# ------------------------------------------------------------
# Process video frame-by-frame
# ------------------------------------------------------------

completed = False
try:
    for _ in tqdm(range(frame_count)):

        ret, frame = cap.read()

        if not ret:
            break

        # --------------------------------------------------------
        # YOLO inference
        # --------------------------------------------------------

        results = model(frame, conf=0.2, device=DEVICE)

        result = results[0]

        names = result.names

        # --------------------------------------------------------
        # Iterate detections
        # --------------------------------------------------------

        for box in result.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            conf = float(box.conf[0])

            cls_id = int(box.cls[0])

            class_name = names[cls_id]

            label = class_name

            # ====================================================
            # If detected object is a car -> run CLIP
            # NOTE: Trucks are intentionally excluded from brand
            # classification because the linear probe was trained
            # on car-only images. Truck crops are out-of-distribution
            # and would produce miscalibrated softmax confidences.
            # ====================================================

            if class_name.lower() == "car":

                # Optional size filtering
                if (x2 - x1) > MIN_CROP_SIZE and (y2 - y1) > MIN_CROP_SIZE:

                    # Crop car
                    car_crop = frame[y1:y2, x1:x2]

                    if car_crop.size > 0:

                        try:

                            brand, brand_conf = predict_car_brand(car_crop)

                            print(f"  CLIP: {brand} @ {brand_conf:.3f}  (crop {x2-x1}x{y2-y1})")

                            if brand_conf >= BRAND_CONF_THRESHOLD:
                                label = f"{brand} ({brand_conf:.2f})"
                            # else: keep label = "car"

                        except Exception as e:

                            print(f"CLIP error: {e}")

            # ----------------------------------------------------
            # Draw box + label chip
            # Colour-code traffic lights: green box/chip for green
            # signals, red for red signals, amber for everything else.
            # Substring match covers trafficLight-Green / -GreenLeft / -Red.
            # ----------------------------------------------------

            name_l = class_name.lower()
            if "green" in name_l:
                box_color = GREEN_COLOR
            elif "red" in name_l:
                box_color = RED_COLOR
            else:
                box_color = BOX_COLOR

            draw_label(frame, x1, y1, x2, y2, f"{label} {conf:.2f}", box_color)

        # --------------------------------------------------------
        # Write frame
        # --------------------------------------------------------

        writer.write(frame)

    completed = True
finally:
    # ------------------------------------------------------------
    # Cleanup — always release, and remove a partial output so Step 3
    # does not silently consume a corrupt annotated_video.mp4.
    # ------------------------------------------------------------
    cap.release()
    writer.release()
    if not completed and output_video_path.is_file():
        try:
            output_video_path.unlink()
            print(f"Removed partial output: {output_video_path}")
        except OSError as rm_exc:
            print(f"Warning: could not remove partial output {output_video_path}: {rm_exc}")

print(f"Saved annotated video to:")
print(output_video_path)

FPS: 29.97
Resolution: 960x540
Frames: 2516


  0%|          | 0/2516 [00:00<?, ?it/s]


0: 288x512 4 cars, 1 truck, 4.8ms
Speed: 0.7ms preprocess, 4.8ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)


  0%|          | 1/2516 [00:00<05:37,  7.46it/s]

  CLIP: Volkswagen @ 0.749  (crop 50x52)
  CLIP: Mercedes @ 0.754  (crop 175x100)
  CLIP: Volkswagen @ 0.470  (crop 79x52)

0: 288x512 4 cars, 1 truck, 4.1ms
Speed: 1.0ms preprocess, 4.1ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)
  CLIP: Volkswagen @ 0.818  (crop 50x53)
  CLIP: Mercedes @ 0.722  (crop 175x99)
  CLIP: Audi @ 0.365  (crop 79x51)

0: 288x512 4 cars, 1 truck, 4.1ms
Speed: 0.6ms preprocess, 4.1ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)
  CLIP: Mercedes @ 0.682  (crop 176x100)
  CLIP: Volkswagen @ 0.611  (crop 51x53)
  CLIP: Volkswagen @ 0.436  (crop 80x51)

0: 288x512 4 cars, 1 truck, 4.5ms
Speed: 0.5ms preprocess, 4.5ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)
  CLIP: Mercedes @ 0.717  (crop 176x100)
  CLIP: Volkswagen @ 0.349  (crop 51x53)
  CLIP: Volkswagen @ 0.703  (crop 82x52)
  CLIP: Volkswagen @ 0.214  (crop 31x33)

0: 288x512 4 cars, 1 truck, 4.4ms
Speed: 0.6ms preprocess, 4.4ms inference, 0.4m

...

  CLIP: Fiat @ 0.141  (crop 109x96)

0: 288x512 1 car, 3.9ms
Speed: 0.5ms preprocess, 3.9ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)
  CLIP: Ford @ 0.151  (crop 99x98)

0: 288x512 1 car, 3.8ms
Speed: 0.5ms preprocess, 3.8ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)
  CLIP: Fiat @ 0.233  (crop 87x102)

0: 288x512 1 car, 4.1ms
Speed: 0.6ms preprocess, 4.1ms inference, 0.3ms postprocess per image at shape (1, 3, 288, 512)
  CLIP: Fiat @ 0.245  (crop 74x101)

0: 288x512 1 car, 4.2ms
Speed: 0.6ms preprocess, 4.2ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)
  CLIP: Fiat @ 0.203  (crop 63x109)

0: 288x512 1 car, 4.0ms
Speed: 0.5ms preprocess, 4.0ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)
  CLIP: Fiat @ 0.134  (crop 48x117)

0: 288x512 1 car, 4.1ms
Speed: 0.5ms preprocess, 4.1ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)
  CLIP: Opel @ 0.142  (crop 35x136)

0: 288x512 1 car, 4.1m

100%|█████████▉| 2510/2516 [01:01<00:00, 61.68it/s]


0: 288x512 (no detections), 3.9ms
Speed: 0.5ms preprocess, 3.9ms inference, 0.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 (no detections), 4.1ms
Speed: 0.6ms preprocess, 4.1ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 (no detections), 4.4ms
Speed: 0.6ms preprocess, 4.4ms inference, 0.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 (no detections), 4.2ms
Speed: 0.6ms preprocess, 4.2ms inference, 0.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 (no detections), 4.0ms
Speed: 0.6ms preprocess, 4.0ms inference, 0.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 (no detections), 4.0ms
Speed: 0.6ms preprocess, 4.0ms inference, 0.1ms postprocess per image at shape (1, 3, 288, 512)


100%|██████████| 2516/2516 [01:01<00:00, 41.12it/s]


Saved annotated video to:
runs_output/detect/clip_predict/annotated_video.mp4
